In [ ]:
import csv
import time
import numpy as np
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import os
import json

# Configuration du WebDriver
service = Service(log_path="NUL")
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--log-level=3")
driver = webdriver.Edge(service=service, options=options)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
}

# Variables globales
all_labels = set()
output_file = "market_data.csv"
backup_file = "market_backup.json"
processed_links = set()

def load_backup():
    """Charge les données depuis le fichier de backup si existant."""
    if os.path.exists(backup_file):
        with open(backup_file, 'r', encoding='utf-8') as f:
            backup = json.load(f)
            processed_links.update(backup.get("processed_links", []))
            return backup.get("headers", []), backup.get("all_data", [])
    return [], []

def save_backup(headers, data):
    """Sauvegarde l'état actuel dans un fichier JSON."""
    with open(backup_file, 'w', encoding='utf-8') as f:
        json.dump({
            "headers": headers,
            "all_data": data,
            "processed_links": list(processed_links),
            "all_labels": list(all_labels)
        }, f, ensure_ascii=False)

def get_listing_links():
    """Récupère les liens des annonces non scrapées."""
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "sc-1jge648-0.jZXrfL"))
        )
        elements = driver.find_elements(By.CLASS_NAME, "sc-1jge648-0.jZXrfL")
        return [el.get_attribute("href") for el in elements
                if el.get_attribute("href") and el.get_attribute("href") not in processed_links]
    except Exception as e:
        print(f"Erreur récupération liens: {str(e)}")
        return []

def scrape_page(url):
    """Scrape une annonce et retourne les données structurées."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, "html.parser")
        data = {}

        # Extraction des caractéristiques
        containers = soup.find_all('div', class_='sc-19cngu6-1')
        for container in containers:
            label = container.find('span', class_='bXFCIH')
            value = container.find('span', class_='fjZBup')
            if label and value:
                clean_label = label.text.strip()
                data[clean_label] = value.text.strip()
                all_labels.add(clean_label)

        # Extraction du prix
        price = soup.select_one('.sc-1x0vz2r-0.lnEFFR')
        if price:
            data["Prix"] = price.text.strip().replace("DH", "").replace(" ", "")

        return data

    except Exception as e:
        print(f"Erreur sur {url}: {str(e)}")
        return None

# Fonctions utilitaires
def rewrite_csv(filename, headers, data):
    """Réécrit complètement le CSV avec les nouveaux headers."""
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(data)

def append_to_csv(filename, headers, row):
    """Ajoute une ligne au CSV existant."""
    file_exists = os.path.exists(filename) and os.path.getsize(filename) > 0
    with open(filename, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(headers)
        writer.writerow(row)

# Initialisation
headers, all_data = load_backup()
driver.get("https://portail-auto-maroc.local/annonces")
time.sleep(3)

try:
    with open(output_file, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)

        # Écriture des headers si fichier vide
        if os.path.getsize(output_file) == 0 and headers:
            writer.writerow(headers)

        for page in range(500):  # Nombre maximal de pages à scraper (à ajuster si nécessaire)
            links = get_listing_links()
            print(f"Page {page+1} - {len(links)} nouveaux liens")

            for link in links:
                data = scrape_page(link)
                if not data:
                    continue

                # Mise à jour des headers si nouveaux champs
                current_headers = sorted(all_labels | {"Prix"})
                if not headers or set(current_headers) != set(headers):
                    headers = current_headers
                    rewrite_csv(output_file, headers, all_data)

                # Ajout des données
                row = [data.get(h, np.nan) for h in headers]
                all_data.append(row)
                processed_links.add(link)

                # Sauvegarde incrémentale
                save_backup(headers, all_data)
                append_to_csv(output_file, headers, row)
                print(f"✓ {link[:60]}...")

            # Pagination
            try:
                wait = WebDriverWait(driver, 10)
                # Locate the last "Next" button using XPath
                next_button = wait.until(
                    EC.element_to_be_clickable((By.XPATH, "(//a[contains(@class, 'sc-2y0ggl-1') and contains(@class, 'yRCEb')])[last()]"))
                )
                next_button.click()
                print(f"✅ Clicked Next Page ({page + 2})")
                time.sleep(1)  # Small delay before next iteration
            except (NoSuchElementException, TimeoutException) as e:
                print(f"❌ Erreur lors du passage à la page suivante: {e}")
                break

            print(f"Page {page + 1} terminée. Total annonces scrapées: {len(all_data)}")

except Exception as e:
    print(f"Une erreur inattendue s'est produite : {e}")

finally:
    driver.quit()
    if os.path.exists(backup_file):
        os.remove(backup_file)
    print(f"Scraping terminé. Données sauvegardées dans {output_file}")